# T01 — Dataset loading and data-engineering pipeline

## Purpose and inherited constraints

This is the primary human-readable implementation, execution, verification, and interpretation record for GitHub Issue #2. It converts the manifest-selected canonical CRITEO-UPLIFTv2.1 CSV into its accepted analytical Parquet derivative and does not create an independent data source of truth.

Inherited constraints are fail-closed: `X = f0…f11`, `T = treatment`, primary `Y = conversion`, `visit` is secondary, `exposure` is audit-only, `_source_row_id` is metadata and never a feature, primary feature precision is `float64`, all released rows are retained, and D23 is `50K → 500K → 2M → full`. This notebook does **not** train a model, construct the frozen split, or access a held-out partition.

Historical decision-evidence runs and their pre-decision status text remain unchanged under `outputs/runs/` and in Git history. Before T01, `data/processed/criteo-uplift-v2.1.parquet` was a manually created Snappy derivative used only for exploratory shape inspection; it was not an authoritative T01 production artifact and the owner removed it. Earlier T01 production verification runs truthfully used `data/processed/criteo-uplift-v2.1-t01.parquet` as a temporary naming convention and their immutable evidence is not rewritten.

The corrected production lifecycle uses the stable canonical consumer path `data/processed/criteo-uplift-v2.1.parquet`. `T01` identifies the task that creates and verifies the derivative, not the dataset's durable filename. This execution creates a new immutable run rather than changing prior run evidence.

## Accepted T01 decisions

| ID | Accepted implementation rule |
|---|---|
| T01-D01 | PyArrow Dataset/Scanner is the low-level loading path; Pandas materialization is explicit and operation-specific. |
| T01-D02 | Semantic identity is mandatory. Physical byte identity is checked only for a same-code/config/package/environment rerun. |
| T01-D03 | `_source_row_id` is the zero-based canonical CSV data-row ordinal and is excluded from `X`. |
| T01-D04 | Resource suitability uses observed operation-specific failures; there is no fixed RAM-percentage gate. |
| T01-D05 | The mutable ignored local selector is separate from the immutable normalized run snapshot. |
| T01-D06 | Production Parquet uses ZSTD and `row_group_size = 1,048,576`. |

DuckDB and Polars are not evaluated because no accepted-path failure has triggered D20. The code below keeps the orchestration and all consequential checks visible; `src/data.py` contains only reusable, regression-sensitive I/O mechanics needed by later tasks.

In [1]:
from __future__ import annotations

import copy
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import threading
import time
from datetime import datetime, timezone
from pathlib import Path

import nbformat
import numpy as np
import pandas as pd
import psutil
import pyarrow as pa
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "docs" / "decision_register.csv").is_file():
            return candidate
    raise RuntimeError("Repository root not found")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import (
    FEATURE_COLUMNS,
    FORBIDDEN_MODEL_COLUMNS,
    MODEL_FEATURES,
    PROCESSED_COLUMNS,
    RAW_COLUMNS,
    ROW_GROUP_SIZE,
    DataContractError,
    assert_model_feature_contract,
    promote_processed_with_rollback,
    convert_csv_to_parquet,
    finalize_artifact_manifest,
    implementation_environment,
    load_selector,
    materialize_pandas,
    open_processed_dataset,
    portable_repo_path,
    scan_batches,
    sha256_file,
    validate_processed_parquet,
    validate_source_identity,
    verify_expected_file_checksum,
    write_json_new,
)

NOTEBOOK_PATH = REPO_ROOT / "notebooks" / "02_data_engineering_pipeline.ipynb"
SELECTOR_PATH = Path(
    os.environ.get(
        "T01_DATA_MANIFEST",
        REPO_ROOT / "configs" / "data_manifest.json",
    )
)
if not SELECTOR_PATH.is_absolute():
    SELECTOR_PATH = REPO_ROOT / SELECTOR_PATH

run_started_at = datetime.now(timezone.utc)
run_id = "t01_production_" + run_started_at.strftime("%Y%m%dT%H%M%SZ_%f")
RUN_ROOT = REPO_ROOT / "outputs" / "runs" / run_id
AUDIT_DIR = RUN_ROOT / "audit"
TEMP_DIR = RUN_ROOT / "temporary"
RUN_ROOT.mkdir(parents=True, exist_ok=False)
AUDIT_DIR.mkdir()
TEMP_DIR.mkdir()


def git_value(*args: str) -> str | None:
    try:
        return subprocess.check_output(
            ["git", *args],
            cwd=REPO_ROOT,
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
    except Exception:
        return None


def notebook_source_sha256(path: Path) -> str:
    notebook = nbformat.read(path, as_version=4)
    source_only = [
        {"cell_type": cell.cell_type, "source": cell.source}
        for cell in notebook.cells
    ]
    return hashlib.sha256(
        json.dumps(source_only, sort_keys=True, ensure_ascii=False).encode("utf-8")
    ).hexdigest()


def combined_code_sha256() -> dict:
    members = {
        "notebooks/02_data_engineering_pipeline.ipynb#sources": notebook_source_sha256(NOTEBOOK_PATH),
        "src/data.py": sha256_file(REPO_ROOT / "src" / "data.py"),
        "tests/test_data.py": sha256_file(REPO_ROOT / "tests" / "test_data.py"),
    }
    digest = hashlib.sha256(
        json.dumps(members, sort_keys=True).encode("utf-8")
    ).hexdigest()
    return {"sha256": digest, "members": members}


selector = load_selector(SELECTOR_PATH, REPO_ROOT)
source_identity = validate_source_identity(selector)
code_identity = combined_code_sha256()
git_status = git_value("status", "--porcelain=v1") or ""
environment = {
    "run_id": run_id,
    "created_at_utc": run_started_at.isoformat(),
    "platform": platform.platform(),
    "machine": platform.machine(),
    "processor": platform.processor(),
    "logical_cpu_count": psutil.cpu_count(logical=True),
    "physical_cpu_count": psutil.cpu_count(logical=False),
    "total_ram_bytes": psutil.virtual_memory().total,
    "python": sys.version,
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "pyarrow": pa.__version__,
    "psutil": psutil.__version__,
    "git_head": git_value("rev-parse", "HEAD"),
    "git_dirty": bool(git_status),
    "code_identity": code_identity,
    "implementation": implementation_environment(),
}
run_config = {
    "run_id": run_id,
    "purpose": "T01_PRODUCTION_DATA_ENGINEERING",
    "local_selector_path": portable_repo_path(SELECTOR_PATH, REPO_ROOT),
    "selector_sha256_at_run_start": sha256_file(SELECTOR_PATH),
    "d23_scale_progression": [50_000, 500_000, 2_000_000, selector.payload["expected_rows"]],
    "sampling_rule": "nested canonical source prefix for engineering promotion only",
    "conversion": selector.payload["conversion"],
    "operational_budget": selector.payload.get("operational_budget"),
    "fixed_ram_percentage_gate": False,
    "model_training": False,
    "split_construction": False,
    "held_out_access": False,
    "duckdb_or_polars_fallback": False,
}
data_manifest = {
    "manifest_role": "IMMUTABLE_RUN_SNAPSHOT",
    "run_id": run_id,
    "created_at_utc": run_started_at.isoformat(),
    "dataset_name": selector.payload["dataset_name"],
    "release_id": selector.payload["release_id"],
    "selector_sha256_at_run_start": sha256_file(SELECTOR_PATH),
    "source": source_identity,
    "processed_target": {
        "path": portable_repo_path(selector.processed_path, REPO_ROOT),
        "generation_mode": "GENERATED_BY_THIS_RUN",
        "actual_identity_artifact": "audit/t01_validation.json",
    },
    "schema_version": selector.payload["schema_version"],
    "raw_columns": list(RAW_COLUMNS),
    "processed_columns": list(PROCESSED_COLUMNS),
    "feature_columns": list(FEATURE_COLUMNS),
    "forbidden_model_columns": list(FORBIDDEN_MODEL_COLUMNS),
    "source_row_identity": selector.payload["source_row_identity"],
    "conversion": selector.payload["conversion"],
    "code_identity": code_identity,
}
write_json_new(RUN_ROOT, "audit/environment.json", environment)
write_json_new(RUN_ROOT, "audit/run_config.json", run_config)
write_json_new(RUN_ROOT, "audit/data_manifest.json", data_manifest)

display(Markdown(f"""**Run:** `{run_id}`  
**Selector:** `{portable_repo_path(SELECTOR_PATH, REPO_ROOT)}`  
**Approved source validation:** `{source_identity['status']}`  
**Expected rows:** `{selector.payload['expected_rows']:,}`"""))

**Run:** `t01_production_20260812T090206Z_287482`  
**Selector:** `configs/data_manifest.json`  
**Approved source validation:** `PASS`  
**Expected rows:** `13,979,592`

## Production workflow and D23 execution protocol

Each D23 rung independently parses the manifest-selected CSV with explicit `float64`/`int64` types, validates labels and finite features, appends the original zero-based source ordinal, writes ZSTD Parquet with the accepted row-group layout, and scans the result to verify row count, columns, physical schema, precision, row-group layout, compression, source-ID order, null masks, values, and row order.

The rungs are nested canonical prefixes used only as engineering promotion gates; they do not define candidate analytical populations. A failed rung stops later promotion. Resource evidence records process RSS, system-available memory, and pagefile observations without converting a RAM percentage into a gate.

In [2]:
class ResourceMonitor:
    def __init__(self, interval_seconds: float = 0.02):
        self.interval_seconds = interval_seconds
        self.process = psutil.Process(os.getpid())
        self.stop_event = threading.Event()

    def _sample(self) -> None:
        while not self.stop_event.is_set():
            vm = psutil.virtual_memory()
            swap = psutil.swap_memory()
            self.peak_rss = max(self.peak_rss, self.process.memory_info().rss)
            self.minimum_system_available = min(self.minimum_system_available, vm.available)
            self.peak_swap_used = max(self.peak_swap_used, swap.used)
            self.stop_event.wait(self.interval_seconds)

    def __enter__(self):
        vm = psutil.virtual_memory()
        swap = psutil.swap_memory()
        self.baseline_rss = self.process.memory_info().rss
        self.peak_rss = self.baseline_rss
        self.system_total = vm.total
        self.starting_system_available = vm.available
        self.minimum_system_available = vm.available
        self.starting_system_memory_percent = vm.percent
        self.baseline_swap_used = swap.used
        self.peak_swap_used = swap.used
        self.started = time.perf_counter()
        self.thread = threading.Thread(target=self._sample, daemon=True)
        self.thread.start()
        return self

    def __exit__(self, exc_type, exc, traceback):
        self.wall_seconds = time.perf_counter() - self.started
        self.stop_event.set()
        self.thread.join(timeout=2)
        vm = psutil.virtual_memory()
        swap = psutil.swap_memory()
        self.final_rss = self.process.memory_info().rss
        self.minimum_system_available = min(self.minimum_system_available, vm.available)
        self.peak_swap_used = max(self.peak_swap_used, swap.used)

    def record(self) -> dict:
        return {
            "wall_seconds": self.wall_seconds,
            "baseline_rss_bytes": self.baseline_rss,
            "peak_rss_bytes": self.peak_rss,
            "peak_rss_delta_bytes": max(0, self.peak_rss - self.baseline_rss),
            "final_rss_bytes": self.final_rss,
            "system_total_bytes": self.system_total,
            "starting_system_available_bytes": self.starting_system_available,
            "minimum_system_available_bytes": self.minimum_system_available,
            "starting_system_memory_percent_observation": self.starting_system_memory_percent,
            "baseline_pagefile_or_swap_used_bytes": self.baseline_swap_used,
            "peak_pagefile_or_swap_used_bytes": self.peak_swap_used,
            "pagefile_or_swap_used_increase_bytes": max(0, self.peak_swap_used - self.baseline_swap_used),
            "fixed_ram_percentage_gate_applied": False,
        }


expected_rows = selector.payload["expected_rows"]
d23_scales = [50_000, 500_000, 2_000_000, expected_rows]
d23_results = []
full_candidate = None
full_conversion = None
full_validation = None

for scale in d23_scales:
    label = "full" if scale == expected_rows else str(scale)
    candidate = TEMP_DIR / f"d23_{label}_candidate_a.parquet"
    result = {
        "rung": label,
        "requested_rows": scale,
        "status": "NOT_RUN",
        "promotion_rule": "correctness and operation-specific resource completion",
        "fixed_ram_percentage_gate": False,
    }
    conversion = None
    validation = None
    monitor = None
    try:
        with ResourceMonitor() as monitor:
            conversion = convert_csv_to_parquet(
                selector.raw_csv_path,
                candidate,
                row_limit=None if scale == expected_rows else scale,
            )
            validation = validate_processed_parquet(
                candidate,
                expected_rows=scale,
                expected_raw_semantics=conversion["raw_semantics"],
            )
        result.update({
            "status": "PASS",
            "conversion": conversion,
            "validation": validation,
            "resource_observations": monitor.record(),
            "resource_failure_observed": False,
            "operational_budget": selector.payload.get("operational_budget"),
        })
    except (MemoryError, pa.ArrowMemoryError) as exc:
        result.update({
            "status": "FAIL",
            "failure_class": "RESOURCE_FAILURE",
            "exception_type": type(exc).__name__,
            "exception": str(exc),
        })
    except Exception as exc:
        result.update({
            "status": "FAIL",
            "failure_class": "CORRECTNESS_OR_EXECUTION_FAILURE",
            "exception_type": type(exc).__name__,
            "exception": str(exc),
        })
    if conversion is not None and "conversion" not in result:
        result["conversion"] = conversion
    if validation is not None and "validation" not in result:
        result["validation"] = validation
    if monitor is not None and hasattr(monitor, "wall_seconds") and "resource_observations" not in result:
        result["resource_observations"] = monitor.record()
    d23_results.append(result)
    if result["status"] != "PASS":
        break
    if scale == expected_rows:
        full_candidate = candidate
        full_conversion = conversion
        full_validation = validation
    else:
        candidate.unlink()

d23_frame = pd.DataFrame([
    {
        "rung": result["rung"],
        "rows": result["requested_rows"],
        "status": result["status"],
        "wall_seconds": result.get("resource_observations", {}).get("wall_seconds"),
        "peak_rss_gib": result.get("resource_observations", {}).get("peak_rss_bytes", 0) / 2**30,
        "minimum_available_gib": result.get("resource_observations", {}).get("minimum_system_available_bytes", 0) / 2**30,
        "pagefile_increase_gib": result.get("resource_observations", {}).get("pagefile_or_swap_used_increase_bytes", 0) / 2**30,
    }
    for result in d23_results
])
display(d23_frame)
if len(d23_results) != len(d23_scales) or any(result["status"] != "PASS" for result in d23_results):
    write_json_new(RUN_ROOT, "audit/t01_scale_report.json", {
        "run_id": run_id,
        "status": "FAIL",
        "scale_progression": d23_scales,
        "results": d23_results,
    })
    raise RuntimeError("D23 stopped at the first failed rung; later promotion is prohibited")

,rung,rows,status,wall_seconds,peak_rss_gib,minimum_available_gib,pagefile_increase_gib
0,50000,50000,PASS,1.499755,0.987904,1.241985,0.000000
1,500000,500000,PASS,2.264602,2.035599,0.626198,0.877769
2,2000000,2000000,PASS,5.416850,3.458866,0.004112,0.030296
3,full,13979592,PASS,29.185028,4.053402,0.000607,0.281242


## Same-environment physical determinism and production promotion

The full conversion is repeated with the same source hashes, notebook/module sources, interpreter, package versions, writer settings, and process environment. Semantic equality remains mandatory for both files. Byte equality is interpreted only as same-environment physical determinism; it is not a substitute for semantic validation and is not generalized across PyArrow versions or platforms.

Only after both full candidates pass is candidate A promoted with caught-exception rollback protection to the exact manifest-declared canonical path `data/processed/criteo-uplift-v2.1.parquet`. This two-step backup/replacement is not claimed to be a true crash-atomic filesystem replacement. A stale `.previous.tmp` state causes a fail-closed stop for explicit validation and recovery; it is never silently discarded. A failed candidate cannot replace the current canonical derivative. Temporary candidate names are run-scoped implementation details, not consumer paths.

In [3]:
if full_candidate is None or full_conversion is None or full_validation is None:
    raise RuntimeError("Full D23 candidate is unavailable")

repeat_candidate = TEMP_DIR / "d23_full_candidate_b.parquet"
with ResourceMonitor() as repeat_monitor:
    repeat_conversion = convert_csv_to_parquet(
        selector.raw_csv_path,
        repeat_candidate,
        row_limit=None,
    )
    repeat_validation = validate_processed_parquet(
        repeat_candidate,
        expected_rows=expected_rows,
        expected_raw_semantics=repeat_conversion["raw_semantics"],
    )

physical_determinism = {
    "scope": "same process, code identity, source hashes, package versions, writer settings, and declared environment",
    "candidate_a_sha256": full_conversion["file_sha256"],
    "candidate_b_sha256": repeat_conversion["file_sha256"],
    "semantic_identity_a": full_validation["semantic_identity"],
    "semantic_identity_b": repeat_validation["semantic_identity"],
    "raw_semantics_equal": full_conversion["raw_semantics"] == repeat_conversion["raw_semantics"],
    "byte_identical": full_conversion["file_sha256"] == repeat_conversion["file_sha256"],
    "repeat_resource_observations": repeat_monitor.record(),
}
if not all([
    physical_determinism["semantic_identity_a"],
    physical_determinism["semantic_identity_b"],
    physical_determinism["raw_semantics_equal"],
    physical_determinism["byte_identical"],
]):
    raise RuntimeError(f"Full conversion determinism gate failed: {physical_determinism}")

promotion = promote_processed_with_rollback(
    full_candidate,
    selector.processed_path,
    expected_sha256=full_conversion["file_sha256"],
)
repeat_candidate.unlink()
final_validation = validate_processed_parquet(
    selector.processed_path,
    expected_rows=expected_rows,
    expected_raw_semantics=full_conversion["raw_semantics"],
)
if final_validation["file_sha256"] != full_conversion["file_sha256"]:
    raise RuntimeError("Final processed artifact does not match the validated candidate")

display(Markdown(f"""**Semantic identity:** `PASS`  
**Same-environment physical determinism:** `PASS`  
**Processed SHA-256:** `{final_validation['file_sha256']}`  
**Processed size:** `{final_validation['file_size_bytes']:,}` bytes"""))

**Semantic identity:** `PASS`  
**Same-environment physical determinism:** `PASS`  
**Processed SHA-256:** `fd2739bb074a50fa0b6796429fe3745e52f287e8770b2c91fd384fa0c0b54fd8`  
**Processed size:** `208,139,615` bytes

## Loading contract: checksum first, scan next, materialize explicitly

Opening the manifest-selected Parquet is a CONSUMPTION boundary: the reusable loader first verifies the selector-pinned processed SHA-256, then returns a PyArrow Dataset. GENERATION may begin with no processed checksum because the derivative does not yet exist; after validated promotion, its final SHA-256 is recorded in immutable run evidence and pinned into the mutable selector before later consumption. Projection and batching happen through a Scanner. Pandas conversion is a separate named operation with an explicit column set and row limit; it is not an automatic consequence of opening the data.

The demonstration below projects two features plus row identity in one Arrow batch, then explicitly materializes only the 50K feature matrix. It does not load treatment, outcomes, `visit`, or `exposure` into `X`.

In [4]:
dataset = open_processed_dataset(selector)
projected_columns = ("f0", "f1", "_source_row_id")
first_projected_batch = next(
    iter(scan_batches(dataset, columns=projected_columns, batch_size=65_536))
)
if tuple(first_projected_batch.schema.names) != projected_columns:
    raise RuntimeError("Projected scanner returned unexpected columns")

assert_model_feature_contract(MODEL_FEATURES)
explicit_feature_frame = materialize_pandas(
    dataset,
    columns=MODEL_FEATURES,
    row_limit=50_000,
)
if list(explicit_feature_frame.columns) != list(FEATURE_COLUMNS):
    raise RuntimeError("Explicit feature materialization violated canonical X order")
if not all(str(dtype) == "float64" for dtype in explicit_feature_frame.dtypes):
    raise RuntimeError("Explicit feature materialization did not preserve float64")
if set(explicit_feature_frame.columns).intersection(FORBIDDEN_MODEL_COLUMNS):
    raise RuntimeError("Forbidden variables entered explicit feature materialization")

loading_contract_evidence = {
    "dataset_type": type(dataset).__name__,
    "implicit_pandas_materialization": False,
    "projected_batch": {
        "columns": list(first_projected_batch.schema.names),
        "rows": first_projected_batch.num_rows,
        "type": type(first_projected_batch).__name__,
    },
    "explicit_pandas_materialization": {
        "declared_columns": list(MODEL_FEATURES),
        "declared_row_limit": 50_000,
        "observed_rows": len(explicit_feature_frame),
        "dtypes": {name: str(dtype) for name, dtype in explicit_feature_frame.dtypes.items()},
        "forbidden_columns_present": [],
    },
}
display(pd.DataFrame([{
    "operation": "Arrow projected batch",
    "rows": first_projected_batch.num_rows,
    "columns": ", ".join(first_projected_batch.schema.names),
}, {
    "operation": "Explicit Pandas materialization",
    "rows": len(explicit_feature_frame),
    "columns": "f0…f11 only",
}]))
del explicit_feature_frame, first_projected_batch

,operation,rows,columns
0,Arrow projected batch,65536,"f0, f1, _source_row_id"
1,Explicit Pandas materialization,50000,f0…f11 only


## Failure-path verification

Synthetic fixtures verify fail-closed behavior without changing the real source: missing selected input, malformed schema, checksum-incompatible/stale artifact, forbidden feature mapping, and attempted mutation of a completed run. These checks complement the automated regression tests; they do not train models or inspect a held-out partition.

In [5]:
FAILURE_FIXTURES = TEMP_DIR / "failure_fixtures"
FAILURE_FIXTURES.mkdir()
failure_results = []


def portable_exception_message(exc: BaseException) -> str:
    message = str(exc)
    message = message.replace(str(RUN_ROOT), "<RUN_ROOT>")
    message = message.replace(str(REPO_ROOT), "<REPO_ROOT>")
    return message


def expect_failure(name: str, operation, expected_exception: type[BaseException]) -> None:
    try:
        operation()
    except expected_exception as exc:
        failure_results.append({
            "check": name,
            "status": "PASS",
            "observed_exception": type(exc).__name__,
            "message": portable_exception_message(exc),
        })
    except Exception as exc:
        failure_results.append({
            "check": name,
            "status": "FAIL",
            "observed_exception": type(exc).__name__,
            "message": portable_exception_message(exc),
        })
    else:
        failure_results.append({
            "check": name,
            "status": "FAIL",
            "observed_exception": None,
            "message": "Expected failure was not raised",
        })


missing_payload = copy.deepcopy(selector.payload)
missing_payload["raw_path"] = "data/raw/does-not-exist.csv"
missing_selector_path = FAILURE_FIXTURES / "missing_selector.json"
missing_selector_path.write_text(json.dumps(missing_payload), encoding="utf-8")
expect_failure(
    "missing_manifest_selected_input",
    lambda: validate_source_identity(load_selector(missing_selector_path, REPO_ROOT)),
    FileNotFoundError,
)

malformed_frame = pd.DataFrame({
    **{name: [0.0] for name in FEATURE_COLUMNS if name != "f11"},
    "treatment": [0],
    "conversion": [0],
    "visit": [0],
    "exposure": [0],
})
malformed_csv = FAILURE_FIXTURES / "malformed.csv"
malformed_frame.to_csv(malformed_csv, index=False)
expect_failure(
    "malformed_schema",
    lambda: convert_csv_to_parquet(
        malformed_csv,
        FAILURE_FIXTURES / "malformed.parquet",
        row_limit=None,
    ),
    DataContractError,
)

expect_failure(
    "stale_or_checksum_incompatible_artifact",
    lambda: verify_expected_file_checksum(selector.processed_path, "0" * 64),
    DataContractError,
)
expect_failure(
    "forbidden_variable_in_model_features",
    lambda: assert_model_feature_contract((*FEATURE_COLUMNS, "treatment")),
    DataContractError,
)

completed_fixture = FAILURE_FIXTURES / "completed_run"
(completed_fixture / "audit").mkdir(parents=True)
(completed_fixture / "audit" / "artifact_manifest.json").write_text(
    json.dumps({"status": "COMPLETED_PASS"}),
    encoding="utf-8",
)
expect_failure(
    "completed_run_immutability",
    lambda: write_json_new(completed_fixture, "audit/late.json", {"late": True}),
    DataContractError,
)

promotion_fixture = FAILURE_FIXTURES / "atomic_promotion"
promotion_fixture.mkdir()
canonical_fixture = promotion_fixture / "canonical.parquet"
failed_candidate_fixture = promotion_fixture / "failed_candidate.parquet"
canonical_fixture.write_bytes(b"existing validated canonical")
failed_candidate_fixture.write_bytes(b"candidate rejected before promotion")
canonical_fixture_hash = sha256_file(canonical_fixture)
expect_failure(
    "failed_candidate_cannot_replace_canonical",
    lambda: promote_processed_with_rollback(
        failed_candidate_fixture,
        canonical_fixture,
        expected_sha256="0" * 64,
    ),
    DataContractError,
)
failure_results[-1]["canonical_unchanged"] = (
    sha256_file(canonical_fixture) == canonical_fixture_hash
)
if not failure_results[-1]["canonical_unchanged"]:
    failure_results[-1]["status"] = "FAIL"

failure_frame = pd.DataFrame(failure_results)
display(failure_frame[["check", "status", "observed_exception"]])
if len(failure_results) != 6 or any(row["status"] != "PASS" for row in failure_results):
    raise RuntimeError("One or more fail-closed verification cases failed")

,check,status,observed_exception
0,missing_manifest_selected_input,PASS,FileNotFoundError
1,malformed_schema,PASS,DataContractError
2,stale_or_checksum_incompatible_artifact,PASS,DataContractError
3,forbidden_variable_in_model_features,PASS,DataContractError
4,completed_run_immutability,PASS,DataContractError
5,failed_candidate_cannot_replace_canonical,PASS,DataContractError


## Evidence closure, artifact linkage, and interpretation

The final cell writes machine-readable evidence before closing the immutable run manifest. `data_manifest.json` is the immutable normalized snapshot of the selector at run start; the actual generated processed identity is recorded in `t01_validation.json`, and both are hashed by `artifact_manifest.json`.

PASS means this declared data-engineering workflow completed and satisfied its correctness gates in this environment. It does not prove causal assumptions, establish a universal memory threshold, validate any model, create a split, or authorize held-out evaluation.

In [6]:
boundary_evidence = {
    "model_training": False,
    "split_construction": False,
    "held_out_access": False,
    "lightgbm_imported": "lightgbm" in sys.modules,
    "sklearn_imported": "sklearn" in sys.modules,
    "model_or_prediction_artifacts_created": any(
        path.name in {"models", "predictions"} for path in RUN_ROOT.iterdir()
    ),
}
if any([
    boundary_evidence["model_training"],
    boundary_evidence["split_construction"],
    boundary_evidence["held_out_access"],
    boundary_evidence["lightgbm_imported"],
    boundary_evidence["sklearn_imported"],
    boundary_evidence["model_or_prediction_artifacts_created"],
]):
    raise RuntimeError(f"T01 execution-boundary violation: {boundary_evidence}")

pagefile_observed = any(
    result.get("resource_observations", {}).get("pagefile_or_swap_used_increase_bytes", 0) > 0
    for result in d23_results
) or physical_determinism["repeat_resource_observations"].get(
    "pagefile_or_swap_used_increase_bytes", 0
) > 0
resource_interpretation = {
    "fixed_ram_percentage_gate": False,
    "oom_or_process_termination_observed": False,
    "incorrect_or_incomplete_execution_observed": False,
    "operational_budget_declared": selector.payload.get("operational_budget") is not None,
    "operational_budget": selector.payload.get("operational_budget"),
    "pagefile_or_swap_increase_observed": pagefile_observed,
    "action": "WARNING" if pagefile_observed else "INFO",
    "interpretation": (
        "Pagefile/swap growth is an investigation observation, not a fixed-percent failure rule."
        if pagefile_observed
        else "No OOM, incomplete execution, declared-budget violation, or observed pagefile/swap growth."
    ),
}

scale_report = {
    "run_id": run_id,
    "status": "PASS",
    "scale_progression": d23_scales,
    "all_rungs_attempted_in_order": [row["requested_rows"] for row in d23_results] == d23_scales,
    "results": d23_results,
    "resource_interpretation": resource_interpretation,
}
validation_report = {
    "run_id": run_id,
    "status": "PASS",
    "source_identity": source_identity,
    "processed_identity": {
        "path": portable_repo_path(selector.processed_path, REPO_ROOT),
        **final_validation,
    },
    "promotion": promotion,
    "physical_determinism": physical_determinism,
    "loading_contract": loading_contract_evidence,
    "model_feature_contract": {
        "X": list(FEATURE_COLUMNS),
        "forbidden": list(FORBIDDEN_MODEL_COLUMNS),
        "status": "PASS",
    },
    "execution_boundaries": boundary_evidence,
}
failure_report = {
    "run_id": run_id,
    "status": "PASS",
    "checks": failure_results,
}
overall_status = "WARN" if resource_interpretation["action"] == "WARNING" else "PASS"
summary = {
    "run_id": run_id,
    "status": overall_status,
    "implementation_status": "T01_IMPLEMENTATION_AND_VERIFICATION_EXECUTED",
    "d23": "PASS",
    "source_and_provenance": "PASS",
    "schema_dtype_row_order_semantic_identity": "PASS",
    "same_environment_physical_determinism": "PASS",
    "failure_paths": "PASS",
    "immutable_manifest_linkage": "PASS",
    "resource_action": resource_interpretation["action"],
    "limitations": [
        "Resource observations apply only to the declared machine and operations.",
        "No universal RAM-percentage threshold is inferred.",
        "Physical determinism is established only for the declared equivalent environment.",
        "Source-row ordinals identify released rows, not people or users.",
        "No causal assumption, model, split, or held-out metric is evaluated by T01.",
    ],
}

write_json_new(RUN_ROOT, "audit/t01_scale_report.json", scale_report)
write_json_new(RUN_ROOT, "audit/t01_validation.json", validation_report)
write_json_new(RUN_ROOT, "audit/t01_failure_paths.json", failure_report)
write_json_new(RUN_ROOT, "audit/t01_summary.json", summary)

if TEMP_DIR.exists():
    shutil.rmtree(TEMP_DIR)

manifest_path = finalize_artifact_manifest(
    RUN_ROOT,
    run_id=run_id,
    external_artifacts=[{
        "path": portable_repo_path(selector.processed_path, REPO_ROOT),
        "role": "manifest_selected_processed_derivative",
        "size_bytes": final_validation["file_size_bytes"],
        "sha256": final_validation["file_sha256"],
        "source_sha256": source_identity["raw_csv"]["sha256"],
        "status": "PASS",
    }],
    final_status=f"COMPLETED_{overall_status}",
    created_at_utc=datetime.now(timezone.utc).isoformat(),
)
artifact_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
manifest_failures = []
for artifact in artifact_manifest["artifacts"]:
    path = RUN_ROOT / artifact["path"]
    if path.stat().st_size != artifact["size_bytes"] or sha256_file(path) != artifact["sha256"]:
        manifest_failures.append(artifact["path"])
required_manifest_paths = {
    "audit/data_manifest.json",
    "audit/environment.json",
    "audit/run_config.json",
    "audit/t01_failure_paths.json",
    "audit/t01_scale_report.json",
    "audit/t01_summary.json",
    "audit/t01_validation.json",
}
observed_manifest_paths = {artifact["path"] for artifact in artifact_manifest["artifacts"]}
if manifest_failures or not required_manifest_paths.issubset(observed_manifest_paths):
    raise RuntimeError(
        f"Artifact-manifest reconciliation failed: hashes={manifest_failures}, "
        f"missing={sorted(required_manifest_paths - observed_manifest_paths)}"
    )

display(Markdown(f"""## Final T01 status: `{overall_status}`

- Run: `{run_id}`
- D23: `PASS` at 50K, 500K, 2M, and full
- Semantic identity and source-row order: `PASS`
- Same-environment physical determinism: `PASS`
- Failure paths and immutable manifest linkage: `PASS`
- Resource action: `{resource_interpretation['action']}`
- Processed artifact: `{portable_repo_path(selector.processed_path, REPO_ROOT)}`
- Artifact manifest: `{manifest_path.relative_to(REPO_ROOT).as_posix()}`"""))
display(d23_frame)

## Final T01 status: `WARN`

- Run: `t01_production_20260812T090206Z_287482`
- D23: `PASS` at 50K, 500K, 2M, and full
- Semantic identity and source-row order: `PASS`
- Same-environment physical determinism: `PASS`
- Failure paths and immutable manifest linkage: `PASS`
- Resource action: `WARNING`
- Processed artifact: `data/processed/criteo-uplift-v2.1.parquet`
- Artifact manifest: `outputs/runs/t01_production_20260812T090206Z_287482/audit/artifact_manifest.json`

,rung,rows,status,wall_seconds,peak_rss_gib,minimum_available_gib,pagefile_increase_gib
0,50000,50000,PASS,1.499755,0.987904,1.241985,0.000000
1,500000,500000,PASS,2.264602,2.035599,0.626198,0.877769
2,2000000,2000000,PASS,5.416850,3.458866,0.004112,0.030296
3,full,13979592,PASS,29.185028,4.053402,0.000607,0.281242
